# Ingestão Bronze - Base dos Dados → S3

Objetivo: extrair as 6 fontes confirmadas na etapa de descoberta (`01_descoberta_fontes.ipynb`)
do BigQuery e gravar como Parquet particionado no S3, na camada Bronze do data lake.

**Origem:** `basedosdados.br_inep_avaliacao_alfabetizacao` (BigQuery)

**Destino:** `s3://brazil-literacy-lakehouse-joaopaulo/bronze/`

In [ ]:
import basedosdados as bd
import pandas as pd

bd.config.billing_project_id = "brazil-literacy-lakehouse"

DATASET_ID = "br_inep_avaliacao_alfabetizacao"

FONTES = {
    "uf": "uf",
    "municipio": "municipio",
    "meta_alfabetizacao_brasil": "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf": "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio": "meta_alfabetizacao_municipio",
    "alunos": "alunos",
}

In [ ]:
query = f"""
SELECT *
FROM `basedosdados.{DATASET_ID}.alunos`
LIMIT 5
"""
bd.read_sql(query, billing_project_id="brazil-literacy-lakehouse")

In [ ]:
import awswrangler as wr

BUCKET = "brazil-literacy-lakehouse-joaopaulo"

resultados_bronze = {}

for nome, tabela in FONTES.items():
    print(f"Extraindo {nome}...")

    query = f"""
    SELECT *
    FROM `basedosdados.{DATASET_ID}.{tabela}`
    """
    df = bd.read_sql(query, billing_project_id="brazil-literacy-lakehouse")

    resposta = wr.s3.to_parquet(
        df=df,
        path=f"s3://{BUCKET}/bronze/{nome}/",
        dataset=True,
        partition_cols=["ano"],
        mode="overwrite",
    )

    resultados_bronze[nome] = resposta
    print(f"  -> {len(df):,} linhas, {len(resposta['paths'])} arquivo(s) gravado(s)")